# BIOMASS TomoSAR — Plotting notebook

**Fourier • Capon • MUSIC • GEDI**

This notebook is dedicated to visualization, with no TomoSAR reconstruction or \(k_z\) interpolation. It loads the saved reconstruction cubes, GEDI, and the BIOMASS geometry; applies a GEDI-based upper-height filter and an optional SKP quality mask; then plots eight figure groups.

**Important:** Every range cut, azimuth cut, and pixel-profile cell creates **two separate Matplotlib figures**: (1) the corrected XP SLC with a red cut line/cross and cyan GEDI tracks, and (2) the tomographic result(s) with GEDI reference. This follows the supplied range-cut example. All output is saved in unique timestamped folders under `outputs_dir()`.

Run **setup sections 1–6** in order (including the SKP quality-mask cell), then run whichever plotting cells you need. Compatible saved MUSIC and Fourier/Capon grids are required. The SLC stack is opened with `mmap_mode="r"` to limit RAM use.
Each plot folder includes `plot_metadata.json` and the same parameter snapshot embedded in each PNG, so filter thresholds, reconstruction IDs, and selected cut positions remain identifiable.


## 1 — Load saved Fourier, Capon, and MUSIC cubes

Select explicit run IDs if needed. Otherwise, load the newest *complete* Fourier/Capon run and newest MUSIC run with identical azimuth, range, and height sample axes. All cubes are memory-mapped.

In [ ]:
import json
from pathlib import Path
import numpy as np

from biomass_tomosar.paths import processed_data_dir

TOMOSAR_RUN_ID = "20261122_213343_042797_UTC"  # e.g. "20261123_150000_123456_UTC"
MUSIC_RUN_ID = "20261126_232602_544538_UTC"    # e.g. "20261127_000000_123456_UTC"

processed_dir = processed_data_dir()


def available_ids(prefix, suffix):
    return {
        p.name[len(prefix):-len(suffix)]
        for p in processed_dir.glob(f"{prefix}*{suffix}")
    }


complete_tomosar = sorted(
    available_ids("fourier_cube_", ".npy")
    & available_ids("capon_cube_", ".npy")
    & available_ids("tomosar_axes_", ".npz")
)
if not complete_tomosar:
    raise FileNotFoundError("No complete Fourier/Capon run with saved axes was found.")

tomosar_run_id = TOMOSAR_RUN_ID or complete_tomosar[-1]
if tomosar_run_id not in complete_tomosar:
    raise FileNotFoundError(f"Fourier/Capon run is missing files: {tomosar_run_id}")

cubeF = np.load(processed_dir / f"fourier_cube_{tomosar_run_id}.npy", mmap_mode="r")
cubeC = np.load(processed_dir / f"capon_cube_{tomosar_run_id}.npy", mmap_mode="r")

with np.load(processed_dir / f"tomosar_axes_{tomosar_run_id}.npz") as f:
    azax = f["azimuth_pixel_centers"].astype(int)
    rgax = f["range_pixel_centers"].astype(int)
    zaxis = f["height_m"].copy()


def music_grid_matches(run_id):
    with np.load(processed_dir / f"music_axes_{run_id}.npz") as f:
        return (
            np.array_equal(f["azimuth_pixel_centers"], azax)
            and np.array_equal(f["range_pixel_centers"], rgax)
            and np.array_equal(f["height_m"], zaxis)
        )


complete_music = sorted(
    available_ids("music_cube_", ".npy")
    & available_ids("music_axes_", ".npz")
)
if MUSIC_RUN_ID is not None:
    if MUSIC_RUN_ID not in complete_music or not music_grid_matches(MUSIC_RUN_ID):
        raise ValueError("Requested MUSIC run is absent or has an incompatible grid.")
    music_run_id = MUSIC_RUN_ID
else:
    compatible_music = [rid for rid in complete_music if music_grid_matches(rid)]
    if not compatible_music:
        raise FileNotFoundError("No saved MUSIC cube has the same grid as Fourier/Capon.")
    music_run_id = compatible_music[-1]

cubeM = np.load(processed_dir / f"music_cube_{music_run_id}.npy", mmap_mode="r")

expected_shape = (len(azax), len(rgax), len(zaxis))
for method, cube in [("Fourier", cubeF), ("Capon", cubeC), ("MUSIC", cubeM)]:
    if cube.shape != expected_shape:
        raise ValueError(f"{method}: cube shape {cube.shape} != {expected_shape}")

tomosar_meta_path = processed_dir / f"tomosar_metadata_{tomosar_run_id}.json"
tomosar_metadata = {}
if tomosar_meta_path.exists():
    with tomosar_meta_path.open(encoding="utf-8") as f:
        tomosar_metadata = json.load(f)

music_meta_path = processed_dir / f"music_metadata_{music_run_id}.json"
music_metadata = {}
if music_meta_path.exists():
    with music_meta_path.open() as f:
        music_metadata = json.load(f)
N_SIGNALS = music_metadata.get("n_signal_sources")
MUSIC_LABEL = f"MUSIC (Ns={N_SIGNALS})" if N_SIGNALS is not None else "MUSIC"

print(f"Fourier/Capon run: {tomosar_run_id}")
print(f"MUSIC run:         {music_run_id}")
print(f"MUSIC N_SIGNALS:   {N_SIGNALS}")
print(f"All cubes:        {expected_shape}")
print(f"Height interval:  {zaxis[0]}–{zaxis[-1]} m")

## 2 — Load GEDI canopy heights and raw GEDI measurements

The gridded GEDI reference uses `gedi_forestHeight.npy` and its `.grd` metadata. The raw point measurements (`gedi.csv`) are retained for overlay on cuts, following your reference figure.

In [ ]:
import pandas as pd
from biomass_tomosar.paths import gedi_data_dir

gedi_dir = gedi_data_dir()

gedi_height = np.load(gedi_dir / "gedi_forestHeight.npy", mmap_mode="r")
grid = pd.read_csv(gedi_dir / "gedi_forestHeight.grd").iloc[0]

az0, naz, daz = float(grid["az0"]), int(grid["naz"]), float(grid["daz"])
rg0, nrg, drg = float(grid["rg0"]), int(grid["nrg"]), float(grid["drg"])

gedi_az = az0 + daz * np.arange(naz)
gedi_rg = rg0 + drg * np.arange(nrg)
assert gedi_height.shape == (naz, nrg)

gedi_points = pd.read_csv(
    gedi_dir / "gedi.csv",
    usecols=["rg", "az", "rh_98", "rh_0"],
)
gedi_points["forest_height"] = gedi_points["rh_98"] - gedi_points["rh_0"]
gedi_points = gedi_points.loc[
    np.isfinite(gedi_points["rg"])
    & np.isfinite(gedi_points["az"])
    & np.isfinite(gedi_points["forest_height"])
    & (gedi_points["forest_height"] >= 0)
].copy()

print("GEDI map:", gedi_height.shape)
print("Raw valid GEDI shots:", len(gedi_points))
print(f"GEDI range: {gedi_rg[0]/1000:.3f}–{gedi_rg[-1]/1000:.3f} km")

## 3 — Initialize `bmStack`, load corrected XP SLC, and restore SAR coordinates

Uses the acquisition order in `xp_stack_metadata.json` when present. `bmStack` reads BIOMASS annotations/LUT descriptions; the full-resolution corrected XP array is opened from `xp_stack.npy` by memory mapping, **without recomputing SKP or flattening corrections**.

In [ ]:
from biomass_tomosar.paths import biomass_stack_dir
from biomass_tomosar.bm_data_loader import bmStack

stack_dir = biomass_stack_dir()
slc_metadata_path = processed_dir / "xp_stack_metadata.json"

if slc_metadata_path.exists():
    with slc_metadata_path.open() as f:
        xp_metadata = json.load(f)
    products = [stack_dir / name for name in xp_metadata["acquisitions"]]
    MASTER_IDX = int(xp_metadata["reference_acquisition_index"])
else:
    products = sorted(p for p in stack_dir.glob("BIO*") if p.is_dir())
    MASTER_IDX = len(products) // 2
    print("Warning: XP acquisition metadata absent; using sorted BIOMASS product order.")

if not products or not all(p.is_dir() for p in products):
    raise FileNotFoundError("Missing BIOMASS acquisition product folder(s).")

stack = bmStack([str(p) for p in products])
xp_stack = np.load(processed_dir / "xp_stack.npy", mmap_mode="r")
n_acq, n_az, n_rg = xp_stack.shape

if n_acq != len(products):
    raise ValueError("Corrected XP stack and BIOMASS product counts differ.")
if not (0 <= MASTER_IDX < n_acq):
    raise ValueError("Invalid reference acquisition index.")

annot = stack.annot_coreg[MASTER_IDX]


def xml_float(name):
    nodes = annot.xpath(f"//*[local-name()='sarImage']/*[local-name()='{name}']")
    if not nodes or nodes[0].text is None:
        raise ValueError(f"Missing SAR annotation: {name}")
    return float(nodes[0].text)


C = 299_792_458.0
az_spacing = xml_float("azimuthPixelSpacing")
rg_spacing = xml_float("rangePixelSpacing")
first_range_time = xml_float("firstSampleSlantRangeTime")

sar_az = np.arange(n_az) * az_spacing
sar_rg = C * first_range_time / 2 + np.arange(n_rg) * rg_spacing

tomo_az = sar_az[azax]
tomo_rg = sar_rg[rgax]

assert cubeF.shape == cubeC.shape == cubeM.shape
assert cubeF.shape == (len(tomo_az), len(tomo_rg), len(zaxis))

print(f"BIOMASS acquisitions: {n_acq} | master index: {MASTER_IDX}")
print(f"Corrected XP SLC:     {xp_stack.shape} (memory-mapped)")
print(f"SAR pixel spacing:    {az_spacing:.3f} m (az), {rg_spacing:.3f} m (rg)")
print(f"TomoSAR coordinates:  {len(tomo_az)} az × {len(tomo_rg)} rg")

## 4 — Plotting setup

Configure both map filters and all plotting options here. The SKP/GEDI masks affect *only peak-height maps*, never cuts or SLC images.


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from datetime import datetime, timezone
from biomass_tomosar.paths import outputs_dir

# =========================================================
# 1. SETTINGS
# =========================================================
ACQ_IDX = 0
SLC_DISPLAY_STEP = 6
GEDI_TRACK_DISPLAY_STEP = 10
SHOW_GEDI_TRACKS = False        # Cyan GEDI shots on SLC overview
SHOW_RAW_GEDI_POINTS = False     # Lime GEDI shots on tomogram cuts
SHOW_LEGENDS = False           # Applies to all helper-generated plots

RANGE_CUT_KM = 768.0    # As in supplied example; None = overlap center
AZIMUTH_CUT_KM = 80   # None = overlap center
PIXEL_AZ_KM = 40
PIXEL_RG_KM = 765

NORMALIZE_CUT_PROFILES = True     # Reference range-cut default
NORMALIZE_PIXEL_PROFILES = True   # Better for comparing spectral shapes
DYNAMIC_RANGE_DB = 20

# MUSIC-specific peak selection
MUSIC_MAX_PEAK_HEIGHT_M = 40.0

# Filters affect ONLY derived 2D/3D peak-height maps, never cuts.
APPLY_GEDI_HEIGHT_FILTER = True
HEIGHT_MARGIN_M = 5.0

SAVE_FIGURES = True
FIGURE_DPI = 200

# SKP phase-screen estimation-quality mask: 2D/3D height maps ONLY.
APPLY_SKP_QUALITY_MASK = True
SKP_QUALITY_THRESHOLD = 0.10   # 0..1; require >= threshold in EVERY secondary

# Save a plot_metadata.json file AND embed the same metadata in each PNG.
EMBED_METADATA_IN_PNG = True

HEIGHT_MIN = 0
HEIGHT_MAX = float(np.max(zaxis))  # Updated by height-map cell
SURFACE_MAX_POINTS = 300
SURFACE_ELEV = 28
SURFACE_AZIM = -65
HEIGHT_CMAP = "viridis"
TOMOGRAM_CMAP = "magma"

if not (0 <= ACQ_IDX < n_acq):
    raise ValueError("ACQ_IDX is outside the SLC acquisition stack.")

# =========================================================
# 2. SAVE EACH PLOT CELL IN A NEW FOLDER
# =========================================================
def save_plot_set(folder_name, figures, selection=None):
    """Save figures, a JSON parameter snapshot, and PNG text metadata.

    Parameters
    ----------
    folder_name : str
        Type of plot (range cut, pixel profile, 3D, ...).
    figures : dict[str, matplotlib.figure.Figure]
        Each figure saves as its own PNG; the SLC remains separate.
    selection : dict, optional
        Actual selected cut position / pixel, not only the requested position.
    """
    if not SAVE_FIGURES:
        return None

    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f_UTC")
    run_dir = outputs_dir() / f"{folder_name}_{stamp}"
    run_dir.mkdir(parents=True, exist_ok=False)

    quality_mask = globals().get("skp_quality_mask")
    quality_valid_pct = (
        float(100 * quality_mask.mean()) if quality_mask is not None else None
    )

    metadata = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "figure_group": folder_name,
        "figures": [f"{stem}.png" for stem in figures],
        "selection": selection or {},
        "data": {
            "tomosar_run_id": tomosar_run_id,
            "music_run_id": music_run_id,
            "music_n_signals": N_SIGNALS,
            "slc_acquisition_index_zero_based": int(ACQ_IDX),
            "reference_acquisition_index_zero_based": int(MASTER_IDX),
            "source_biomass_products": [p.name for p in products],
            "tomo_cube_shape": list(cubeM.shape),
            "height_samples_m": [float(zaxis[0]), float(zaxis[-1]), int(len(zaxis))],
            "fourier_capon_metadata": tomosar_metadata,
            "music_metadata": music_metadata,
        },
        "filters": {
            "gedi_height": {
                "enabled": bool(APPLY_GEDI_HEIGHT_FILTER),
                "gedi_max_height_m": globals().get("GEDI_MAX_HEIGHT"),
                "margin_m": float(HEIGHT_MARGIN_M),
                "threshold_m": globals().get("HEIGHT_THRESHOLD"),
                "applies_to": "2D and 3D peak-height maps only",
            },
            "skp_quality": {
                "enabled": bool(APPLY_SKP_QUALITY_MASK),
                "threshold": float(SKP_QUALITY_THRESHOLD),
                "aggregation": "minimum quality over all non-reference acquisitions",
                "checked_acquisition_indices_zero_based": list(
                    globals().get("skp_checked_acquisitions", [])
                ),
                "percent_spatial_pixels_retained": quality_valid_pct,
                "applies_to": "2D and 3D peak-height maps only; not SLC, GEDI, or cuts",
                "sampling": "BIOMASS LUT quality interpolated at TomoSAR multilook centers",
            },
        },
        "plot_settings": {
            "requested_range_cut_km": RANGE_CUT_KM,
            "requested_azimuth_cut_km": AZIMUTH_CUT_KM,
            "requested_pixel_azimuth_km": PIXEL_AZ_KM,
            "requested_pixel_range_km": PIXEL_RG_KM,
            "normalize_cut_profiles": bool(NORMALIZE_CUT_PROFILES),
            "normalize_pixel_profiles": bool(NORMALIZE_PIXEL_PROFILES),
            "dynamic_range_db": float(DYNAMIC_RANGE_DB),
            "slc_display_step": int(SLC_DISPLAY_STEP),
            "gedi_track_display_step": int(GEDI_TRACK_DISPLAY_STEP),
            "show_gedi_tracks_on_slc": bool(SHOW_GEDI_TRACKS),
            "show_raw_gedi_on_cuts": bool(SHOW_RAW_GEDI_POINTS),
            "show_legends": bool(SHOW_LEGENDS),
            "height_display_min_m": float(HEIGHT_MIN),
            "height_display_max_m": float(HEIGHT_MAX),
            "surface_max_points_per_axis": int(SURFACE_MAX_POINTS),
            "surface_elevation_deg": float(SURFACE_ELEV),
            "surface_azimuth_deg": float(SURFACE_AZIM),
            "height_colormap": HEIGHT_CMAP,
            "tomogram_colormap": TOMOGRAM_CMAP,
            "figure_dpi": int(FIGURE_DPI),
        },
    }

    metadata_json = json.dumps(metadata, indent=2, ensure_ascii=False)
    with (run_dir / "plot_metadata.json").open("x", encoding="utf-8") as f:
        f.write(metadata_json + "\n")

    png_metadata = (
        {"Software": "biomass_tomosar plotting notebook",
         "Description": metadata_json}
        if EMBED_METADATA_IN_PNG else None
    )
    for filename_stem, fig in figures.items():
        path = run_dir / f"{filename_stem}.png"
        with path.open("xb") as f:
            fig.savefig(
                f, format="png", dpi=FIGURE_DPI,
                bbox_inches="tight", metadata=png_metadata,
            )
        print("Saved:", path)
    print("Parameters:", run_dir / "plot_metadata.json")
    return run_dir

# =========================================================
# 3. SPATIAL SELECTION (OVERLAP-AWARE)
# =========================================================
def choose_cut(tomo_axis_m, gedi_axis_m, requested_km, name):
    lo = max(float(min(tomo_axis_m)), float(min(gedi_axis_m)))
    hi = min(float(max(tomo_axis_m)), float(max(gedi_axis_m)))
    if lo >= hi:
        raise ValueError(f"No TomoSAR/GEDI overlap on {name} axis.")
    target = (lo + hi) / 2 if requested_km is None else float(requested_km) * 1000
    if not (lo <= target <= hi):
        raise ValueError(
            f"{name} = {target/1000:.3f} km outside common coverage "
            f"[{lo/1000:.3f}, {hi/1000:.3f}] km"
        )
    idx = int(np.argmin(np.abs(tomo_axis_m - target)))
    return idx, float(tomo_axis_m[idx])


def select_range_cut():
    return choose_cut(tomo_rg, gedi_rg, RANGE_CUT_KM, "Range")


def select_azimuth_cut():
    return choose_cut(tomo_az, gedi_az, AZIMUTH_CUT_KM, "Azimuth")


def select_pixel():
    ia, az_m = choose_cut(tomo_az, gedi_az, PIXEL_AZ_KM, "Pixel azimuth")
    ir, rg_m = choose_cut(tomo_rg, gedi_rg, PIXEL_RG_KM, "Pixel range")
    return ia, ir, az_m, rg_m

# =========================================================
# 4. SLC DISPLAY (SEPARATE FIGURE)
# =========================================================
slc_sample = np.asarray(
    xp_stack[ACQ_IDX, ::SLC_DISPLAY_STEP, ::SLC_DISPLAY_STEP]
)
slc_amplitude = np.abs(slc_sample)
slc_db = 20 * np.log10(np.maximum(slc_amplitude, 1e-8))
slc_good = slc_db[np.isfinite(slc_db) & (slc_amplitude > 0)]
slc_vmin, slc_vmax = (
    np.percentile(slc_good, [2, 98]) if slc_good.size else (-40.0, 0.0)
)
slc_extent = [
    sar_rg[0]/1000, sar_rg[-1]/1000,
    sar_az[0]/1000, sar_az[-1]/1000,
]


def make_slc_figure(kind, az_m=None, rg_m=None, points=None):
    """Independent SLC figure. For kind="points", pass (id, az_m, rg_m) triples."""
    fig, ax = plt.subplots(figsize=(9, 11), constrained_layout=True)
    im = ax.imshow(
        slc_db, extent=slc_extent, origin="lower", aspect="auto",
        cmap="gray", vmin=slc_vmin, vmax=slc_vmax,
    )
    if kind == "range":
        ax.axvline(rg_m/1000, color="red", linewidth=2,
                   label=f"Range cut: {rg_m/1000:.3f} km")
    elif kind == "azimuth":
        ax.axhline(az_m/1000, color="red", linewidth=2,
                   label=f"Azimuth cut: {az_m/1000:.3f} km")
    elif kind == "pixel":
        ax.plot(rg_m/1000, az_m/1000, marker="x", color="red",
                markersize=14, markeredgewidth=3, linestyle="none",
                label="Selected TomoSAR pixel", zorder=5)
    elif kind == "points":
        if not points:
            raise ValueError("Pass points=[(point_id, az_m, rg_m), ...].")
        for point_id, point_az_m, point_rg_m in points:
            ax.plot(point_rg_m/1000, point_az_m/1000,
                    marker="x", color="red", markersize=13,
                    markeredgewidth=2.5, linestyle="none", zorder=5)
            ax.annotate(str(point_id), (point_rg_m/1000, point_az_m/1000),
                        xytext=(8, -8), textcoords="offset points", fontsize=10,
                        color="red", weight="bold", zorder=6)
    else:
        raise ValueError(f"Unknown SLC annotation: {kind}")

    if SHOW_GEDI_TRACKS:
        pts = gedi_points.iloc[::GEDI_TRACK_DISPLAY_STEP]
        ax.scatter(
            pts["rg"]/1000, pts["az"]/1000,
            s=1, color="cyan", alpha=0.25, rasterized=True,
            label="GEDI tracks",
        )
    ax.set_title(f"Corrected XP SLC — Acquisition {ACQ_IDX + 1}")
    ax.set_xlabel("Slant range [km]")
    ax.set_ylabel("Azimuth [km]")
    ax.set_xlim(slc_extent[0], slc_extent[1])
    ax.set_ylim(slc_extent[3], slc_extent[2])
    if SHOW_LEGENDS:
        ax.legend(loc="upper right")
    fig.colorbar(im, ax=ax, label="SLC amplitude [dB]", shrink=0.7)
    return fig

# =========================================================
# 5. TOMOGRAM CUT WITH BOTH GRIDDED AND RAW GEDI
# =========================================================
def prepare_spectrum_db(section, normalize=False):
    """Convert (space, height) section to (height, space) dB."""
    power = np.asarray(section, dtype=np.float64)
    valid = np.isfinite(power) & (power > 0)
    if normalize:
        peaks = np.max(np.where(valid, power, 0), axis=1, keepdims=True)
        out = np.full_like(power, np.nan)
        np.divide(power, peaks, out=out, where=valid & (peaks > 0))
        power = out
        valid = np.isfinite(power) & (power > 0)
    db = np.full(power.shape, np.nan, dtype=np.float64)
    db[valid] = 10 * np.log10(power[valid])
    return db.T


def color_limits(db, normalize):
    if normalize:
        return -DYNAMIC_RANGE_DB, 0.0
    good = db[np.isfinite(db)]
    vmax = float(np.percentile(good, 99)) if good.size else 0.0
    return vmax - DYNAMIC_RANGE_DB, vmax


def clean_gedi_heights(heights):
    h = np.ma.filled(np.ma.asarray(heights, dtype=float), np.nan).copy()
    h[~np.isfinite(h) | (h < 0)] = np.nan
    return h


def plot_tomogram(ax, cube, kind, idx, title, spectrum_label):
    """Range: height-vs-azimuth; azimuth: height-vs-range."""
    if kind == "range":
        section = cube[:, idx, :]
        x_m = tomo_az
        cut_m = tomo_rg[idx]
        gedi_idx = int(np.argmin(np.abs(gedi_rg - cut_m)))
        gedi_x = gedi_az
        gedi_h = clean_gedi_heights(gedi_height[:, gedi_idx])
        tomo_step = float(np.median(np.diff(tomo_rg)))
        half_width = max(30.0, tomo_step / 2)
        near_cut = gedi_points.loc[
            np.abs(gedi_points["rg"] - cut_m) <= half_width
        ]
        point_axis = "az"
        x_label = "Azimuth [km]"
    elif kind == "azimuth":
        section = cube[idx, :, :]
        x_m = tomo_rg
        cut_m = tomo_az[idx]
        gedi_idx = int(np.argmin(np.abs(gedi_az - cut_m)))
        gedi_x = gedi_rg
        gedi_h = clean_gedi_heights(gedi_height[gedi_idx, :])
        tomo_step = float(np.median(np.diff(tomo_az)))
        half_width = max(30.0, tomo_step / 2)
        near_cut = gedi_points.loc[
            np.abs(gedi_points["az"] - cut_m) <= half_width
        ]
        point_axis = "rg"
        x_label = "Slant range [km]"
    else:
        raise ValueError("kind must be 'range' or 'azimuth'")

    db = prepare_spectrum_db(section, normalize=NORMALIZE_CUT_PROFILES)
    vmin, vmax = color_limits(db, NORMALIZE_CUT_PROFILES)
    im = ax.pcolormesh(
        x_m/1000, zaxis, db, shading="nearest",
        cmap=TOMOGRAM_CMAP, vmin=vmin, vmax=vmax,
        rasterized=True,
    )
    ax.plot(
        gedi_x/1000, gedi_h, color="cyan", linewidth=1.8,
        label="GEDI canopy height", zorder=4,
    )
    if SHOW_RAW_GEDI_POINTS and not near_cut.empty:
        ax.scatter(
            near_cut[point_axis]/1000,
            near_cut["forest_height"],
            s=8, color="lime", alpha=0.55,
            label="Raw GEDI", rasterized=True, zorder=5,
        )
    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel("Height [m]")
    ax.set_xlim(x_m[0]/1000, x_m[-1]/1000)
    ax.set_ylim(zaxis[0], zaxis[-1])
    ax.grid(alpha=0.15)
    if SHOW_LEGENDS:
        ax.legend(loc="upper right")
    return im, (
        "Relative spectrum [dB]" if NORMALIZE_CUT_PROFILES
        else spectrum_label
    )

# =========================================================
# 6. PIXEL PROFILES AND GEDI REFERENCE
# =========================================================
def pixel_gedi_height(az_m, rg_m):
    ga = int(np.argmin(np.abs(gedi_az - az_m)))
    gr = int(np.argmin(np.abs(gedi_rg - rg_m)))
    height = float(np.ma.filled(np.ma.asarray(gedi_height[ga, gr]), np.nan))
    return height if np.isfinite(height) and height >= 0 else None


def pixel_spectrum_db(cube, ia, ir):
    p = np.asarray(cube[ia, ir, :], dtype=np.float64)
    good = np.isfinite(p) & (p > 0)
    p = np.where(good, p, np.nan)
    if NORMALIZE_PIXEL_PROFILES and np.any(good):
        p = p / np.nanmax(p)
    with np.errstate(divide="ignore", invalid="ignore"):
        return 10 * np.log10(p)


def make_pixel_profile_figure(methods, ia, ir, az_m, rg_m):
    """methods is a list of (label, cube, color)."""
    fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
    for label, cube, color in methods:
        db = pixel_spectrum_db(cube, ia, ir)
        ax.plot(zaxis, db, label=label, linewidth=2, color=color)

    gedi_h = pixel_gedi_height(az_m, rg_m)
    if gedi_h is not None:
        ax.axvline(
            gedi_h, color="green", linestyle="--", linewidth=2,
            label=f"GEDI canopy height: {gedi_h:.1f} m",
        )
    ax.set_title(f"Vertical profile — Az {az_m/1000:.3f} km, Rg {rg_m/1000:.3f} km")
    ax.set_xlabel("Height [m]")
    ax.set_ylabel(
        "Relative spectrum [dB]" if NORMALIZE_PIXEL_PROFILES
        else "Spectrum [dB] (method-dependent units)"
    )
    ax.set_xlim(zaxis[0], zaxis[-1])
    if NORMALIZE_PIXEL_PROFILES:
        ax.set_ylim(-DYNAMIC_RANGE_DB, 2)
    ax.grid(alpha=0.3)
    if SHOW_LEGENDS:
        ax.legend(loc="best")
    return fig

# =========================================================
# 7. 3D PEAK HEIGHT SURFACES
# =========================================================
def plot_height_surface(ax, height_map, az_m, rg_m, title):
    step_a = max(1, int(np.ceil(len(az_m) / SURFACE_MAX_POINTS)))
    step_r = max(1, int(np.ceil(len(rg_m) / SURFACE_MAX_POINTS)))
    z = clean_gedi_heights(height_map[::step_a, ::step_r])
    z[(z < HEIGHT_MIN) | (z > HEIGHT_MAX)] = np.nan
    x, y = np.meshgrid(rg_m[::step_r]/1000, az_m[::step_a]/1000)
    ax.plot_surface(
        x, y, z, cmap=HEIGHT_CMAP,
        vmin=HEIGHT_MIN, vmax=HEIGHT_MAX,
        linewidth=0, antialiased=False,
        rcount=z.shape[0], ccount=z.shape[1],
    )
    ax.set_title(title, pad=12)
    ax.set_xlabel("Slant range [km]")
    ax.set_ylabel("Azimuth [km]")
    ax.set_zlabel("Height [m]")
    ax.set_xlim(tomo_rg[0]/1000, tomo_rg[-1]/1000)
    ax.set_ylim(tomo_az[-1]/1000, tomo_az[0]/1000)
    ax.set_zlim(HEIGHT_MIN, HEIGHT_MAX)
    ax.view_init(elev=SURFACE_ELEV, azim=SURFACE_AZIM)
    ax.set_box_aspect((1, 1.3, 0.65))


def add_height_colorbar(fig, axes):
    norm = mpl.colors.Normalize(HEIGHT_MIN, HEIGHT_MAX)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=HEIGHT_CMAP)
    sm.set_array([])
    fig.colorbar(sm, ax=axes, label="Height [m]", shrink=0.62, pad=0.08)

print("Plotting helpers ready.")
print(f"GEDI filter: {APPLY_GEDI_HEIGHT_FILTER} | margin: {HEIGHT_MARGIN_M:.1f} m")
print(f"SKP filter (maps only): {APPLY_SKP_QUALITY_MASK} | quality: {SKP_QUALITY_THRESHOLD:.2f}")
print("Run any plot cell below; SLC figures are always separate.")

## 5 — Calculate SKP quality mask (for height maps only)

This cell samples the quality LUTs and creates `skp_quality_mask`; it does not modify any cube, height map or cut.


In [ ]:
import numpy as np
from scipy.interpolate import RegularGridInterpolator
from tqdm.auto import tqdm

# =========================================================
# SKP QUALITY MASK (settings in preceding setup cell)
# =========================================================
# Quality range: 0 (worst) to 1 (best), -9999/missing = invalid.
# Strict rule: ALL secondary acquisitions must have quality >= 0.10
# at the multilooked pixel center. The master is not checked.
# This cell only COMPUTES skp_quality_mask.
# The height-map cell applies the mask; cuts remain unmasked.

if not (0 <= SKP_QUALITY_THRESHOLD <= 1):
    raise ValueError("SKP_QUALITY_THRESHOLD must be between 0 and 1.")

expected_grid = (len(azax), len(rgax))
skp_checked_acquisitions = []
skp_min_quality = None
skp_quality_mask = np.ones(expected_grid, dtype=bool)

if APPLY_SKP_QUALITY_MASK:
    if len(stack.luts) != n_acq:
        raise ValueError("BIOMASS LUT/acquisition counts do not match.")

    skp_min_quality = np.full(expected_grid, np.inf, dtype=np.float32)
    for i, lut in enumerate(tqdm(stack.luts, desc="Sampling SKP quality LUTs")):
        if i == MASTER_IDX:
            continue  # the reference acquisition has no reference-secondary phase

        try:
            var = lut.get(
                "skpPhaseCalibration", "skpCalibrationPhaseScreenQuality"
            ).nc
            quality_native = np.ma.filled(
                np.ma.asarray(var[:]).astype(np.float32), np.nan
            )
        except (KeyError, IndexError, AttributeError) as exc:
            raise ValueError(
                f"SKP quality LUT missing for acquisition {i}. "
                "Disable APPLY_SKP_QUALITY_MASK or verify SKP calibration."
            ) from exc

        # Convert the BIOMASS no-data sentinel and out-of-range values
        # to NaN BEFORE interpolation (conservative around invalid areas).
        quality_native[
            ~np.isfinite(quality_native)
            | (quality_native < 0)
            | (quality_native > 1)
        ] = np.nan

        # Mirror bmStack's linear interpolation, evaluating ONLY at
        # TomoSAR center pixels instead of upsampling the entire SLC image.
        interpolator = RegularGridInterpolator(
            (np.asarray(lut._azLut), np.asarray(lut._rgLut)),
            quality_native, method="linear", bounds_error=False,
            fill_value=np.nan,
        )
        az_target = np.asarray(lut._az)[azax]
        rg_target = np.asarray(lut._rg)[rgax]
        az_mesh, rg_mesh = np.meshgrid(az_target, rg_target, indexing="ij")
        q = interpolator(
            np.column_stack((az_mesh.ravel(), rg_mesh.ravel()))
        ).reshape(expected_grid)

        q[~np.isfinite(q) | (q < 0) | (q > 1)] = np.nan
        skp_min_quality = np.minimum(skp_min_quality, q)
        skp_checked_acquisitions.append(int(i))

    if not skp_checked_acquisitions:
        raise ValueError("No secondary acquisitions available for SKP quality mask.")

    skp_quality_mask = (
        np.isfinite(skp_min_quality)
        & (skp_min_quality >= SKP_QUALITY_THRESHOLD)
    )

print(
    f"SKP quality filter: {'ON' if APPLY_SKP_QUALITY_MASK else 'OFF'} "
    f"| threshold = {SKP_QUALITY_THRESHOLD:.2f}"
)
print("Secondary acquisitions checked:", skp_checked_acquisitions)
print(
    f"Spatial pixels retained: {skp_quality_mask.sum():,} / "
    f"{skp_quality_mask.size:,} "
    f"({100 * skp_quality_mask.mean():.1f}%)"
)


## 6 — Generate 2D/3D peak-height maps with optional GEDI and SKP filtering

This cell finds each profile's original global maximum. A peak above the GEDI threshold or failing SKP is assigned NaN. It never falls back to a second peak.


In [ ]:
# =========================================================
# PEAK-HEIGHT MAPS — GEDI + SKP FILTERS (2D/3D ONLY)
# =========================================================
import numpy as np
from scipy.signal import find_peaks

# =========================================================
# 1. MUSIC PEAK-SELECTION SETTINGS
# =========================================================
MUSIC_MAX_PEAK_HEIGHT_M = 40.0

# =========================================================
# 2. GEDI HEIGHT THRESHOLD
# =========================================================
valid_gedi = np.ma.filled(
    np.ma.asarray(gedi_height, dtype=float),
    np.nan
)

valid_gedi = valid_gedi[
    np.isfinite(valid_gedi) & (valid_gedi >= 0)
]

if valid_gedi.size == 0:
    raise ValueError("No valid GEDI canopy heights available.")

GEDI_MAX_HEIGHT = float(valid_gedi.max())
HEIGHT_THRESHOLD = GEDI_MAX_HEIGHT + HEIGHT_MARGIN_M

# =========================================================
# 3. VALIDATE INPUTS AND SKP MASK
# =========================================================
if not (cubeF.shape == cubeC.shape == cubeM.shape):
    raise ValueError("Fourier, Capon and MUSIC cubes have different shapes.")

z = np.asarray(zaxis)

if cubeF.shape[-1] != len(z):
    raise ValueError("Height axis and cube dimensions do not match.")

if APPLY_SKP_QUALITY_MASK:
    if "skp_quality_mask" not in globals():
        raise RuntimeError(
            "Run the SKP quality-mask cell before this cell."
        )

    if skp_quality_mask.shape != cubeF.shape[:2]:
        raise ValueError(
            "SKP mask and TomoSAR grid have different shapes."
        )

# =========================================================
# 4. PEAK-HEIGHT EXTRACTION
# =========================================================
def peak_height_map(cube, method="argmax", chunk_size=32):
    """
    Extract peak heights from TomoSAR spectra.

    method="argmax":
        Select the global maximum.
        Used for Fourier and Capon.

    method="music":
        Detect local maxima using scipy.signal.find_peaks.
        Select the highest-elevation local maximum satisfying:
            height <= MUSIC_MAX_PEAK_HEIGHT_M

        If no valid peak exists, return NaN for the pixel.

    GEDI and SKP filters act on the resulting height maps only.
    Original reconstruction cubes remain unchanged.
    """

    if method not in ("argmax", "music"):
        raise ValueError(
            "method must be 'argmax' or 'music'"
        )

    result = np.full(
        cube.shape[:2],
        np.nan,
        dtype=np.float32
    )

    # Process cube in spatial chunks to limit memory use
    for i in range(0, cube.shape[0], chunk_size):

        j = min(i + chunk_size, cube.shape[0])

        power = np.asarray(
            cube[i:j],
            dtype=np.float64
        )

        valid = np.isfinite(power) & (power > 0)

        heights = np.full(
            power.shape[:2],
            np.nan,
            dtype=np.float32
        )

        # -------------------------------------------------
        # FOURIER / CAPON — GLOBAL MAXIMUM
        # -------------------------------------------------
        if method == "argmax":

            peak_idx = np.argmax(
                np.where(valid, power, -np.inf),
                axis=2
            )

            heights = np.asarray(
                z[peak_idx],
                dtype=np.float32
            )

            # Discard pixels without valid power
            heights[~np.any(valid, axis=2)] = np.nan

        # -------------------------------------------------
        # MUSIC — HIGHEST LOCAL PEAK BELOW THRESHOLD
        # -------------------------------------------------
        elif method == "music":

            for ia in range(power.shape[0]):
                for ir in range(power.shape[1]):

                    profile = power[ia, ir, :]
                    profile_valid = valid[ia, ir, :]

                    if np.count_nonzero(profile_valid) < 3:
                        continue

                    # Detect local maxima over the full spectrum
                    clean_profile = np.where(
                        profile_valid,
                        profile,
                        -np.inf
                    )

                    peaks, _ = find_peaks(clean_profile)

                    if peaks.size == 0:
                        continue

                    # Exclude peaks adjacent to invalid samples
                    peaks = peaks[
                        profile_valid[peaks - 1]
                        & profile_valid[peaks + 1]
                    ]

                    # Keep only peaks below the elevation limit
                    peaks = peaks[
                        z[peaks] <= MUSIC_MAX_PEAK_HEIGHT_M
                    ]

                    if peaks.size == 0:
                        continue

                    # Select the peak at the highest elevation,
                    # regardless of its amplitude
                    selected_peak = peaks[
                        np.argmax(z[peaks])
                    ]

                    heights[ia, ir] = z[selected_peak]

        # -------------------------------------------------
        # OPTIONAL GEDI HEIGHT FILTER
        # -------------------------------------------------
        if APPLY_GEDI_HEIGHT_FILTER:
            heights[
                heights > HEIGHT_THRESHOLD
            ] = np.nan

        result[i:j] = heights

    # -----------------------------------------------------
    # OPTIONAL SKP QUALITY MASK — HEIGHT MAPS ONLY
    # -----------------------------------------------------
    if APPLY_SKP_QUALITY_MASK:
        result[~skp_quality_mask] = np.nan

    return result


# =========================================================
# 5. COMPUTE HEIGHT MAPS
# =========================================================

# Fourier: global maximum
fourier_height_map = peak_height_map(
    cubeF,
    method="argmax"
)

# Capon: global maximum
capon_height_map = peak_height_map(
    cubeC,
    method="argmax"
)

# MUSIC: highest local peak below 40 m
music_height_map = peak_height_map(
    cubeM,
    method="music"
)

# =========================================================
# 6. DISPLAY HEIGHT LIMIT
# =========================================================
# This affects only the display, not reconstructed spectra.

HEIGHT_MAX = (
    min(float(np.max(zaxis)), HEIGHT_THRESHOLD)
    if APPLY_GEDI_HEIGHT_FILTER
    else float(np.max(zaxis))
)

# =========================================================
# 7. SUMMARY
# =========================================================
print(f"GEDI max: {GEDI_MAX_HEIGHT:.2f} m")
print(f"GEDI threshold: {HEIGHT_THRESHOLD:.2f} m")

print(
    f"MUSIC maximum allowed peak height: "
    f"{MUSIC_MAX_PEAK_HEIGHT_M:.2f} m"
)

print(
    f"GEDI filter: "
    f"{'ON' if APPLY_GEDI_HEIGHT_FILTER else 'OFF'}"
)

print(
    f"SKP map filter: "
    f"{'ON' if APPLY_SKP_QUALITY_MASK else 'OFF'}"
)

print("\nPeak selection:")
print("Fourier : Global maximum")
print("Capon   : Global maximum")
print("MUSIC   : Highest local peak below elevation threshold")

print("\nRetained heights:")

for label, hmap in [
    ("Fourier", fourier_height_map),
    ("Capon", capon_height_map),
    ("MUSIC", music_height_map),
]:
    print(
        f"{label:8}: "
        f"{np.isfinite(hmap).sum():,} / "
        f"{hmap.size:,} heights retained"
    )

---
# A — MUSIC only

Each cut/profile creates a **separate SLC figure**, followed by its MUSIC-only plot. The 3D figure compares MUSIC peak scattering height against GEDI canopy height.

## 7 — MUSIC range cut vs GEDI (+ separate SLC)

In [ ]:
RG_IDX, cut_rg_m = select_range_cut()
print(f"MUSIC range cut at {cut_rg_m/1000:.3f} km (index {RG_IDX})")

fig_slc = make_slc_figure("range", rg_m=cut_rg_m)

fig_tomo, ax = plt.subplots(figsize=(15, 6), constrained_layout=True)
im, label = plot_tomogram(
    ax, cubeM, "range", RG_IDX,
    f"{MUSIC_LABEL} — Range {cut_rg_m/1000:.3f} km",
    "MUSIC pseudo-spectrum [dB]",
)
fig_tomo.colorbar(im, ax=ax, label=label, shrink=0.85)
fig_tomo.suptitle("MUSIC TomoSAR vs GEDI — range cut", fontsize=14)

save_plot_set("music_range_cut", {
    "slc_overview": fig_slc,
    "music_gedi_range_cut": fig_tomo,
}, selection={
    "cut_type": "range",
    "tomo_range_index": int(RG_IDX),
    "actual_range_km": float(cut_rg_m / 1000),
})
plt.show()

## 8 — MUSIC azimuth cut vs GEDI (+ separate SLC)

In [ ]:
AZ_IDX, cut_az_m = select_azimuth_cut()
print(f"MUSIC azimuth cut at {cut_az_m/1000:.3f} km (index {AZ_IDX})")

fig_slc = make_slc_figure("azimuth", az_m=cut_az_m)

fig_tomo, ax = plt.subplots(figsize=(15, 6), constrained_layout=True)
im, label = plot_tomogram(
    ax, cubeM, "azimuth", AZ_IDX,
    f"{MUSIC_LABEL} — Azimuth {cut_az_m/1000:.3f} km",
    "MUSIC pseudo-spectrum [dB]",
)
fig_tomo.colorbar(im, ax=ax, label=label, shrink=0.85)
fig_tomo.suptitle("MUSIC TomoSAR vs GEDI — azimuth cut", fontsize=14)

save_plot_set("music_azimuth_cut", {
    "slc_overview": fig_slc,
    "music_gedi_azimuth_cut": fig_tomo,
}, selection={
    "cut_type": "azimuth",
    "tomo_azimuth_index": int(AZ_IDX),
    "actual_azimuth_km": float(cut_az_m / 1000),
})
plt.show()

## 9 — MUSIC pixel profile vs GEDI (+ separate SLC)

In [ ]:
ia, ir, az_m, rg_m = select_pixel()
print(f"Selected pixel: az {az_m/1000:.3f} km, range {rg_m/1000:.3f} km")

fig_slc = make_slc_figure("pixel", az_m=az_m, rg_m=rg_m)
fig_profile = make_pixel_profile_figure(
    [(MUSIC_LABEL, cubeM, "tab:purple")], ia, ir, az_m, rg_m
)

profile = np.asarray(cubeM[ia, ir, :])
valid = np.isfinite(profile) & (profile > 0)
if np.any(valid):
    idx = np.argmax(np.where(valid, profile, -np.inf))
    print(f"MUSIC peak: {zaxis[idx]:.1f} m")
print("GEDI canopy height:", pixel_gedi_height(az_m, rg_m))

save_plot_set("music_pixel_profile", {
    "slc_overview": fig_slc,
    "music_gedi_pixel_profile": fig_profile,
}, selection={
    "cut_type": "pixel",
    "tomo_azimuth_index": int(ia),
    "tomo_range_index": int(ir),
    "actual_azimuth_km": float(az_m / 1000),
    "actual_range_km": float(rg_m / 1000),
})
plt.show()

## 10 — MUSIC vs GEDI 3D height surfaces

The MUSIC surface uses `music_height_map` from Cell 4, not the raw cube. The GEDI-derived upper height filter applies to the MUSIC **peak-height map**. No SLC overlay is needed for a height-map view.

In [ ]:
fig_3d = plt.figure(figsize=(16, 8), constrained_layout=True)
axes = [fig_3d.add_subplot(1, 2, i+1, projection="3d") for i in range(2)]

plot_height_surface(
    axes[0], music_height_map, tomo_az, tomo_rg,
    f"{MUSIC_LABEL} — Peak scattering height",
)
plot_height_surface(
    axes[1], gedi_height, gedi_az, gedi_rg,
    "GEDI — Canopy height",
)
add_height_colorbar(fig_3d, axes)
fig_3d.suptitle("MUSIC and GEDI — 3D height maps", fontsize=15)

save_plot_set(
    "music_vs_gedi_3d", {"music_vs_gedi_3d": fig_3d},
    selection={"plot_type": "3d_height_maps", "methods": ["MUSIC", "GEDI"]},
)
plt.show()

---
# B — Fourier + Capon + MUSIC comparisons

For range and azimuth cuts, the **SLC overview is its own figure**. The second figure contains three vertically stacked tomograms with matching spatial coordinates, each with a separate colorbar and its own interpolated/raw GEDI overlays (as in the supplied reference cell). For the pixel cut, all three normalized profiles share one axis.

## 11 — Three-method range cut vs GEDI (+ separate SLC)

In [ ]:
RG_IDX, cut_rg_m = select_range_cut()
fig_slc = make_slc_figure("range", rg_m=cut_rg_m)

fig_tomo, axes = plt.subplots(
    3, 1, figsize=(15, 14), sharex=True, sharey=True,
    constrained_layout=True,
)
methods = [
    ("Fourier TomoSAR", cubeF, "Fourier power [dB]"),
    ("Capon TomoSAR", cubeC, "Capon power [dB]"),
    (f"{MUSIC_LABEL} TomoSAR", cubeM, "MUSIC pseudo-spectrum [dB]"),
]
for ax, (name, cube, unit) in zip(axes, methods):
    im, label = plot_tomogram(ax, cube, "range", RG_IDX, name, unit)
    fig_tomo.colorbar(im, ax=ax, label=label, shrink=0.85)

mode = "Normalized" if NORMALIZE_CUT_PROFILES else "Unnormalized"
fig_tomo.suptitle(
    f"Fourier / Capon / MUSIC — {mode} spectra\n"
    f"Range: {cut_rg_m/1000:.3f} km",
    fontsize=15,
)

save_plot_set("comparison_range_cut", {
    "slc_overview": fig_slc,
    "fourier_capon_music_gedi_range_cut": fig_tomo,
}, selection={
    "cut_type": "range",
    "tomo_range_index": int(RG_IDX),
    "actual_range_km": float(cut_rg_m / 1000),
})
plt.show()

## 12 — Three-method azimuth cut vs GEDI (+ separate SLC)

In [ ]:
AZ_IDX, cut_az_m = select_azimuth_cut()
fig_slc = make_slc_figure("azimuth", az_m=cut_az_m)

fig_tomo, axes = plt.subplots(
    3, 1, figsize=(15, 14), sharex=True, sharey=True,
    constrained_layout=True,
)
methods = [
    ("Fourier TomoSAR", cubeF, "Fourier power [dB]"),
    ("Capon TomoSAR", cubeC, "Capon power [dB]"),
    (f"{MUSIC_LABEL} TomoSAR", cubeM, "MUSIC pseudo-spectrum [dB]"),
]
for ax, (name, cube, unit) in zip(axes, methods):
    im, label = plot_tomogram(ax, cube, "azimuth", AZ_IDX, name, unit)
    fig_tomo.colorbar(im, ax=ax, label=label, shrink=0.85)

mode = "Normalized" if NORMALIZE_CUT_PROFILES else "Unnormalized"
fig_tomo.suptitle(
    f"Fourier / Capon / MUSIC — {mode} spectra\n"
    f"Azimuth: {cut_az_m/1000:.3f} km",
    fontsize=15,
)

save_plot_set("comparison_azimuth_cut", {
    "slc_overview": fig_slc,
    "fourier_capon_music_gedi_azimuth_cut": fig_tomo,
}, selection={
    "cut_type": "azimuth",
    "tomo_azimuth_index": int(AZ_IDX),
    "actual_azimuth_km": float(cut_az_m / 1000),
})
plt.show()

## 13 — Three-method pixel profile vs GEDI (+ separate SLC)

Default: each method's profile is normalized to its own maximum (0 dB), making peak positions and widths directly comparable despite MUSIC having a non-physical pseudo-spectrum amplitude.

In [ ]:
ia, ir, az_m, rg_m = select_pixel()
fig_slc = make_slc_figure("pixel", az_m=az_m, rg_m=rg_m)

methods = [
    ("Fourier", cubeF, "tab:blue"),
    ("Capon", cubeC, "tab:orange"),
    (MUSIC_LABEL, cubeM, "tab:purple"),
]
fig_profile = make_pixel_profile_figure(methods, ia, ir, az_m, rg_m)

for name, cube, _ in methods:
    prof = np.asarray(cube[ia, ir, :])
    valid = np.isfinite(prof) & (prof > 0)
    if np.any(valid):
        idx = np.argmax(np.where(valid, prof, -np.inf))
        print(f"{name:15} peak: {zaxis[idx]:.1f} m")
print("GEDI canopy height:", pixel_gedi_height(az_m, rg_m))

save_plot_set("comparison_pixel_profile", {
    "slc_overview": fig_slc,
    "fourier_capon_music_gedi_pixel_profile": fig_profile,
}, selection={
    "cut_type": "pixel",
    "tomo_azimuth_index": int(ia),
    "tomo_range_index": int(ir),
    "actual_azimuth_km": float(az_m / 1000),
    "actual_range_km": float(rg_m / 1000),
})
plt.show()

## 14 — Four 3D height surfaces in a 2×2 layout

**Top row:** Fourier and Capon. **Bottom row:** MUSIC and GEDI. All use the same height color scale (`0` to `max(GEDI)+5 m`). The three radar surfaces represent **dominant scattering height**, while GEDI represents canopy height above its ground reference.

In [ ]:
fig_3d = plt.figure(figsize=(18, 14), constrained_layout=True)
axes = [fig_3d.add_subplot(2, 2, i+1, projection="3d") for i in range(4)]

plot_height_surface(
    axes[0], fourier_height_map, tomo_az, tomo_rg,
    "Fourier — Peak scattering height",
)
plot_height_surface(
    axes[1], capon_height_map, tomo_az, tomo_rg,
    "Capon — Peak scattering height",
)
plot_height_surface(
    axes[2], music_height_map, tomo_az, tomo_rg,
    f"{MUSIC_LABEL} — Peak scattering height",
)
plot_height_surface(
    axes[3], gedi_height, gedi_az, gedi_rg,
    "GEDI — Canopy height",
)
add_height_colorbar(fig_3d, axes)
fig_3d.suptitle(
    "BIOMASS TomoSAR — Fourier, Capon, MUSIC and GEDI",
    fontsize=16,
)

save_plot_set(
    "comparison_3d", {"fourier_capon_music_gedi_3d": fig_3d},
    selection={"plot_type": "3d_height_maps",
               "methods": ["Fourier", "Capon", "MUSIC", "GEDI"]},
)
plt.show()

---
### Notes on interpretation

- **Fourier / Capon:** reconstructed radar power. **MUSIC:** pseudo-spectrum, not a calibrated scattering power estimate; the panels intentionally use different color limits when not normalized.
- **GEDI:** `rh_98 - rh_0` canopy height; it need not coincide with the strongest P-band XP scattering height.
- **GEDI-based filter:** applied only to the 2D/3D peak-height maps. The original profiles and cut tomograms remain untouched.
- **Output organization:** every plotting cell saves its outputs in a unique folder under the configured `outputs` directory. Every range, azimuth, and pixel folder contains *two files*, one SLC overview and one tomogram/profile figure.
- **SKP mask:** Optional quality threshold applied across all secondary acquisitions, at TomoSAR pixel centers. Invalid samples are excluded. The SLC overview and GEDI references are not masked.
- **Figure provenance:** Every save creates a `plot_metadata.json` file and embeds the parameters in PNG `Description` metadata.


## 15 — Multi-point Fourier and Capon threshold exploration

One SLC overview with all numbered crosses. Separate Fourier and Capon figures per selected pixel. No GEDI-height or SKP filtering on cuts; no legends. All figures/CSV saved under the `outputs` path from TOML.


In [ ]:
# =========================================================
# FOURIER / CAPON / MUSIC — MULTI-POINT PIXEL CUTS
# =========================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import find_peaks
from IPython.display import display

# =========================================================
# 1. SETTINGS
# =========================================================
# 4 azimuth positions × 5 range positions = 20 pixels
az_values = np.linspace(5, 40, 4)
rg_values = np.linspace(765, 770, 5)

PIXEL_COORDS_KM = [
    (float(az), float(rg))
    for az in az_values
    for rg in rg_values
]

PROFILE_DYNAMIC_RANGE_DB = 30
PROFILE_XLIM = (float(zaxis[0]), float(zaxis[-1]))
SHOW_THRESHOLD_TEXTBOX = True

# MUSIC: select the highest local peak below this height.
# Reuses the setting from the peak-height-map cell, if defined.
MUSIC_MAX_PEAK_HEIGHT_M = float(
    globals().get("MUSIC_MAX_PEAK_HEIGHT_M", 40.0)
)

# =========================================================
# 2. VALIDATION
# =========================================================
if not PIXEL_COORDS_KM:
    raise ValueError("No pixel coordinates specified.")

if not (cubeF.shape == cubeC.shape == cubeM.shape):
    raise ValueError("Fourier, Capon and MUSIC cubes do not match.")

z = np.asarray(zaxis, dtype=float)

if cubeF.shape[-1] != len(z):
    raise ValueError("Cube height dimension does not match zaxis.")

if not np.all(np.diff(z) > 0):
    raise ValueError("zaxis must be strictly increasing.")

methods = [
    ("Fourier", cubeF, "tab:blue"),
    ("Capon", cubeC, "tab:orange"),
    ("MUSIC", cubeM, "tab:purple"),
]

records = []
summary_rows = []

# =========================================================
# 3. EXTRACT PROFILES AT SELECTED PIXELS
# =========================================================
for point_id, (az_req_km, rg_req_km) in enumerate(
    PIXEL_COORDS_KM, start=1
):

    try:
        ia, az_m = choose_cut(
            tomo_az, gedi_az, az_req_km, "Azimuth"
        )

        ir, rg_m = choose_cut(
            tomo_rg, gedi_rg, rg_req_km, "Range"
        )

    except ValueError as exc:
        print(f"Point {point_id} skipped: {exc}")
        continue

    gedi_h = pixel_gedi_height(az_m, rg_m)
    profiles = {}

    for name, cube, color in methods:

        power = np.asarray(
            cube[ia, ir, :],
            dtype=np.float64
        )

        valid = np.isfinite(power) & (power > 0)

        power = np.where(valid, power, np.nan)

        if not np.any(valid):
            profiles[name] = None
            continue

        # -------------------------------------------------
        # 3.1 GLOBAL MAXIMUM — FOR NORMALIZATION ONLY
        # -------------------------------------------------
        global_idx = int(np.nanargmax(power))
        global_power = float(power[global_idx])

        with np.errstate(divide="ignore", invalid="ignore"):
            db = 10 * np.log10(power / global_power)

        # -------------------------------------------------
        # 3.2 SELECT THE PEAK HEIGHT
        # -------------------------------------------------
        selected_idx = None

        if name in ("Fourier", "Capon"):

            # Global maximum
            selected_idx = global_idx

        elif name == "MUSIC":

            # Detect local maxima in the full MUSIC spectrum
            clean_power = np.where(
                valid, power, -np.inf
            )

            peaks, _ = find_peaks(clean_power)

            # Reject peaks bordering invalid samples
            peaks = peaks[
                valid[peaks - 1]
                & valid[peaks + 1]
            ]

            # Keep peaks below the elevation limit
            allowed_peaks = peaks[
                z[peaks] <= MUSIC_MAX_PEAK_HEIGHT_M
            ]

            if allowed_peaks.size > 0:

                # Select highest ELEVATION, not highest POWER
                selected_idx = int(
                    allowed_peaks[
                        np.argmax(z[allowed_peaks])
                    ]
                )

        # -------------------------------------------------
        # 3.3 SELECTED PEAK PROPERTIES
        # -------------------------------------------------
        peak_height = np.nan
        peak_power = np.nan
        peak_db = np.nan

        if selected_idx is not None:

            peak_height = float(z[selected_idx])
            peak_power = float(power[selected_idx])

            # dB level of selected peak, relative to global max
            peak_db = float(db[selected_idx])

        # -------------------------------------------------
        # 3.4 SPECTRUM AT GEDI HEIGHT
        # -------------------------------------------------
        db_at_gedi = np.nan
        power_at_gedi = np.nan

        if gedi_h is not None:

            good = np.isfinite(power) & np.isfinite(z)

            if np.count_nonzero(good) >= 2:

                z_good = z[good]
                power_good = power[good]

                if z_good[0] <= gedi_h <= z_good[-1]:

                    # Interpolate LINEAR spectrum
                    power_at_gedi = float(
                        np.interp(
                            gedi_h,
                            z_good,
                            power_good
                        )
                    )

                    if power_at_gedi > 0:

                        # Normalize to global maximum for display
                        db_at_gedi = float(
                            10 * np.log10(
                                power_at_gedi / global_power
                            )
                        )

        # -------------------------------------------------
        # 3.5 SELECTED-PEAK TO GEDI DIFFERENCE
        # -------------------------------------------------
        # Fourier / Capon: peak_db = 0 dB
        # MUSIC: selected peak may be below 0 dB
        #
        # Delta = 10 log10(P_selected / P_GEDI)

        delta_db = np.nan

        if (
            np.isfinite(peak_db)
            and np.isfinite(db_at_gedi)
        ):
            delta_db = peak_db - db_at_gedi

        profiles[name] = {
            "db": db,
            "color": color,
            "peak_height": peak_height,
            "peak_power": peak_power,
            "peak_db": peak_db,
            "db_at_gedi": db_at_gedi,
            "delta_db": delta_db,
        }

    if all(p is None for p in profiles.values()):
        print(f"Point {point_id}: no valid profiles.")
        continue

    # -----------------------------------------------------
    # STORE PIXEL DATA
    # -----------------------------------------------------
    records.append({
        "id": point_id,
        "az_m": az_m,
        "rg_m": rg_m,
        "az_req_km": float(az_req_km),
        "rg_req_km": float(rg_req_km),
        "ia": ia,
        "ir": ir,
        "gedi_h": gedi_h,
        "profiles": profiles,
    })

    row = {
        "point_id": point_id,
        "azimuth_km": az_m / 1000,
        "range_km": rg_m / 1000,
        "gedi_height_m": gedi_h,
    }

    for name, _, _ in methods:

        p = profiles[name]
        key = name.lower()

        row[f"{key}_peak_height_m"] = (
            p["peak_height"] if p else np.nan
        )

        row[f"{key}_selected_peak_db"] = (
            p["peak_db"] if p else np.nan
        )

        row[f"{key}_db_at_gedi"] = (
            p["db_at_gedi"] if p else np.nan
        )

        row[f"{key}_delta_db"] = (
            p["delta_db"] if p else np.nan
        )

    summary_rows.append(row)

if not records:
    raise ValueError("No valid points to plot.")

# =========================================================
# 4. ONE SHARED SLC WITH ALL SELECTED PIXELS
# =========================================================
fig_slc = make_slc_figure(
    "points",
    points=[
        (r["id"], r["az_m"], r["rg_m"])
        for r in records
    ]
)

fig_slc.axes[0].set_title(
    f"XP SLC — {len(records)} selected pixels"
)

figures = {
    "slc_all_points": fig_slc
}

# =========================================================
# 5. SEPARATE FOURIER, CAPON AND MUSIC FIGURES
# =========================================================
for record in records:

    point_id = record["id"]
    gedi_h = record["gedi_h"]

    for name, _, _ in methods:

        p = record["profiles"][name]

        fig, ax = plt.subplots(
            figsize=(10, 6),
            constrained_layout=True
        )

        # -------------------------------------------------
        # RECONSTRUCTED SPECTRUM
        # -------------------------------------------------
        if p is not None:

            ax.plot(
                z,
                p["db"],
                color=p["color"],
                linewidth=2.2
            )

            # Selected peak:
            # argmax for Fourier/Capon
            # highest local peak <= 40 m for MUSIC
            if np.isfinite(p["peak_height"]):

                ax.axvline(
                    p["peak_height"],
                    color=p["color"],
                    linestyle="--",
                    linewidth=1.8
                )

                # Highlight the selected peak
                ax.scatter(
                    p["peak_height"],
                    p["peak_db"],
                    color=p["color"],
                    s=65,
                    zorder=6
                )

        # -------------------------------------------------
        # GEDI CANOPY HEIGHT
        # -------------------------------------------------
        if gedi_h is not None:

            ax.axvline(
                gedi_h,
                color="green",
                linestyle=":",
                linewidth=2.2
            )

            if (
                p is not None
                and np.isfinite(p["db_at_gedi"])
            ):

                ax.scatter(
                    gedi_h,
                    p["db_at_gedi"],
                    color="green",
                    s=60,
                    zorder=5
                )

        # -------------------------------------------------
        # THRESHOLD TEXTBOX
        # -------------------------------------------------
        if SHOW_THRESHOLD_TEXTBOX:

            lines = []

            if gedi_h is not None:
                lines.append(
                    f"GEDI canopy: {gedi_h:.1f} m"
                )

            if p is not None:

                if np.isfinite(p["peak_height"]):

                    peak_label = (
                        "Highest local peak"
                        if name == "MUSIC"
                        else "Global maximum"
                    )

                    lines.append(
                        f"{peak_label}: {p['peak_height']:.1f} m"
                    )

                    if name == "MUSIC":
                        lines.append(
                            f"Selected peak: {p['peak_db']:.2f} dB"
                        )

                elif name == "MUSIC":
                    lines.append(
                        f"No local peak below "
                        f"{MUSIC_MAX_PEAK_HEIGHT_M:.1f} m"
                    )

                if np.isfinite(p["delta_db"]):

                    lines.append(
                        f"At GEDI: {p['db_at_gedi']:.2f} dB"
                    )

                    lines.append(
                        f"Selected peak-to-GEDI: "
                        f"{p['delta_db']:.2f} dB"
                    )

            if lines:

                ax.text(
                    0.02,
                    0.03,
                    "\n".join(lines),
                    transform=ax.transAxes,
                    va="bottom",
                    ha="left",
                    fontsize=10,
                    bbox={
                        "boxstyle": "round",
                        "facecolor": "white",
                        "alpha": 0.85
                    }
                )

        # -------------------------------------------------
        # FIGURE FORMATTING
        # -------------------------------------------------
        ax.set_title(
            f"{name} — Point {point_id}\n"
            f"Az {record['az_m']/1000:.3f} km, "
            f"Range {record['rg_m']/1000:.3f} km"
        )

        ax.set_xlabel("Height [m]")
        ax.set_ylabel("Normalized spectrum [dB]")

        ax.set_xlim(*PROFILE_XLIM)
        ax.set_ylim(-PROFILE_DYNAMIC_RANGE_DB, 2)

        ax.grid(alpha=0.3)

        # No legend
        figures[
            f"point_{point_id:02d}_{name.lower()}"
        ] = fig

# =========================================================
# 6. SUMMARY TABLE
# =========================================================
summary_df = pd.DataFrame(summary_rows)

print(
    "Threshold summary "
    "(cuts are NOT filtered by GEDI/SKP):"
)

display(summary_df)

print("\nMedian selected-peak-to-GEDI differences:")

for name, _, _ in methods:

    median_db = summary_df[
        f"{name.lower()}_delta_db"
    ].median()

    print(f"{name:8s}: {median_db:.2f} dB")

print(
    f"\nMUSIC selection: highest local peak "
    f"at or below {MUSIC_MAX_PEAK_HEIGHT_M:.1f} m"
)

# =========================================================
# 7. SAVE ALL FIGURES AND METADATA
# =========================================================
# Uses outputs_dir() from config/local.toml,
# through the existing save_plot_set() helper.

selection = {
    "requested_coordinates_km": [
        list(map(float, xy))
        for xy in PIXEL_COORDS_KM
    ],

    "selected_points": [
        {
            "id": int(r["id"]),
            "azimuth_km": r["az_m"] / 1000,
            "range_km": r["rg_m"] / 1000,
            "gedi_height_m": r["gedi_h"],
        }
        for r in records
    ],

    "algorithms": ["Fourier", "Capon", "MUSIC"],

    "peak_selection": {
        "fourier": "global maximum",
        "capon": "global maximum",
        "music": "highest local peak below elevation limit",
        "music_max_peak_height_m": MUSIC_MAX_PEAK_HEIGHT_M,
    },

    "music_n_signals": globals().get("N_SIGNALS"),

    "height_filter_applied_to_cuts": False,
    "skp_mask_applied_to_cuts": False,

    "normalization": (
        "Each spectrum normalized to its global maximum (0 dB)"
    ),

    "delta_db_definition": (
        "Selected peak dB minus spectrum dB at GEDI height"
    ),

    "profile_dynamic_range_db": float(
        PROFILE_DYNAMIC_RANGE_DB
    ),
}

run_dir = save_plot_set(
    "fourier_capon_music_multipoint",
    figures,
    selection=selection
)

# =========================================================
# 8. SAVE SUMMARY CSV
# =========================================================
if SAVE_FIGURES and run_dir is not None:

    summary_path = run_dir / "threshold_summary.csv"

    summary_df.to_csv(
        summary_path,
        index=False
    )

    print("CSV saved:", summary_path)

# =========================================================
# 9. DISPLAY AND CLOSE
# =========================================================
plt.show()

for fig in figures.values():
    plt.close(fig)

## Tree height Maps

In [ ]:
# =========================================================
# 2D / 3D CANOPY HEIGHT COMPARISON — TOMOSAR VS GEDI
# =========================================================
import numpy as np
import matplotlib.pyplot as plt

from scipy.signal import find_peaks
from scipy.interpolate import RegularGridInterpolator

# =========================================================
# 1. SETTINGS
# =========================================================
DB_THRESHOLDS = {
    "Fourier": 1.96,
    "Capon": 2.51,
    "MUSIC": 5.89,
}

# Highest-elevation local MUSIC peak below this limit
MUSIC_MAX_PEAK_HEIGHT_M = float(
    globals().get("MUSIC_MAX_PEAK_HEIGHT_M", 40.0)
)

# Include MUSIC without threshold for comparison
PLOT_MUSIC_NO_THRESHOLD = True

CHUNK_SIZE = 16

# =========================================================
# 2. VALIDATION
# =========================================================
z = np.asarray(zaxis, dtype=np.float64)

if z.ndim != 1 or len(z) < 3 or not np.all(np.diff(z) > 0):
    raise ValueError("zaxis must be increasing with at least 3 samples.")

if not (cubeF.shape == cubeC.shape == cubeM.shape):
    raise ValueError("TomoSAR cube shapes do not match.")

if cubeF.shape != (len(tomo_az), len(tomo_rg), len(z)):
    raise ValueError("Cube shape does not match TomoSAR axes.")

if APPLY_SKP_QUALITY_MASK:
    if "skp_quality_mask" not in globals():
        raise RuntimeError("Run the SKP quality-mask cell first.")

    if skp_quality_mask.shape != cubeF.shape[:2]:
        raise ValueError("SKP mask dimensions do not match.")

# =========================================================
# 3. HEIGHT EXTRACTION HELPER
# =========================================================
def canopy_height_maps_db(cube, algorithm, db_drops, chunk_size=16):
    """
    Return canopy-height maps for the requested dB drops.

    Fourier / Capon:
        Select global maximum.

    MUSIC:
        Use scipy.signal.find_peaks().
        Select highest-elevation local peak <= MUSIC_MAX_PEAK_HEIGHT_M.

    For a positive dB drop:
        Find the FIRST downward threshold crossing when moving
        from the selected peak toward higher elevation.

    For db_drop=None:
        Return the height of the selected peak itself.

    GEDI/SKP filters are applied to the output maps only.
    """

    maps = {
        drop: np.full(cube.shape[:2], np.nan, dtype=np.float32)
        for drop in db_drops
    }

    nz = len(z)
    height_intervals = np.arange(nz - 1)[None, None, :]

    for start in range(0, cube.shape[0], chunk_size):

        stop = min(start + chunk_size, cube.shape[0])

        power = np.asarray(
            cube[start:stop],
            dtype=np.float64
        )

        valid = np.isfinite(power) & (power > 0)
        shape = power.shape[:2]

        selected_idx = np.full(shape, -1, dtype=np.int32)

        # -------------------------------------------------
        # 3.1 SELECT PEAK
        # -------------------------------------------------
        if algorithm in ("Fourier", "Capon"):

            selected_idx = np.argmax(
                np.where(valid, power, -np.inf),
                axis=2
            ).astype(np.int32)

            selected_idx[~np.any(valid, axis=2)] = -1

        elif algorithm == "MUSIC":

            for ia in range(shape[0]):
                for ir in range(shape[1]):

                    profile = power[ia, ir]
                    profile_valid = valid[ia, ir]

                    if np.count_nonzero(profile_valid) < 3:
                        continue

                    peaks, _ = find_peaks(
                        np.where(profile_valid, profile, -np.inf)
                    )

                    # Exclude invalid neighbouring samples
                    peaks = peaks[
                        profile_valid[peaks - 1]
                        & profile_valid[peaks + 1]
                    ]

                    # Highest qualifying peak in elevation.
                    # When three peaks qualify, this selects
                    # the third (uppermost) peak.
                    peaks = peaks[
                        z[peaks] <= MUSIC_MAX_PEAK_HEIGHT_M
                    ]

                    if peaks.size:
                        selected_idx[ia, ir] = int(
                            peaks[np.argmax(z[peaks])]
                        )

        else:
            raise ValueError(f"Unknown algorithm: {algorithm}")

        selected_valid = selected_idx >= 0
        safe_idx = np.maximum(selected_idx, 0)

        selected_power = np.take_along_axis(
            power,
            safe_idx[..., None],
            axis=2
        )[..., 0]

        # Work in dB for crossing interpolation
        power_db = np.full_like(power, np.nan)

        with np.errstate(divide="ignore", invalid="ignore"):
            power_db[valid] = 10 * np.log10(power[valid])
            selected_db = 10 * np.log10(selected_power)

        # -------------------------------------------------
        # 3.2 HEIGHT MAP FOR EACH dB THRESHOLD
        # -------------------------------------------------
        for drop_db in db_drops:

            heights = np.full(shape, np.nan, dtype=np.float64)

            if drop_db is None or drop_db == 0:

                # No threshold: selected peak height
                heights[selected_valid] = z[
                    selected_idx[selected_valid]
                ]

            else:

                # Threshold relative to SELECTED peak,
                # not necessarily MUSIC's global maximum.
                target_db = selected_db - float(drop_db)

                left_db = power_db[:, :, :-1]
                right_db = power_db[:, :, 1:]

                # Only search ABOVE the selected peak.
                # Detect first downward crossing.
                crossings = (
                    selected_valid[..., None]
                    & (height_intervals >= selected_idx[..., None])
                    & np.isfinite(left_db)
                    & np.isfinite(right_db)
                    & (left_db >= target_db[..., None])
                    & (right_db <= target_db[..., None])
                    & (left_db > right_db)
                )

                found = np.any(crossings, axis=2)

                crossing_idx = np.argmax(
                    crossings, axis=2
                )

                left_value = np.take_along_axis(
                    left_db,
                    crossing_idx[..., None],
                    axis=2
                )[..., 0]

                right_value = np.take_along_axis(
                    right_db,
                    crossing_idx[..., None],
                    axis=2
                )[..., 0]

                # Linear interpolation of dB crossing
                with np.errstate(divide="ignore", invalid="ignore"):
                    fraction = (
                        (left_value - target_db)
                        / (left_value - right_value)
                    )

                fraction = np.clip(fraction, 0, 1)

                interpolated_height = (
                    z[crossing_idx]
                    + fraction
                    * (z[crossing_idx + 1] - z[crossing_idx])
                )

                heights[found] = interpolated_height[found]

            # -------------------------------------------------
            # 3.3 OPTIONAL MAP FILTERS
            # -------------------------------------------------
            if APPLY_GEDI_HEIGHT_FILTER:
                heights[heights > HEIGHT_THRESHOLD] = np.nan

            if APPLY_SKP_QUALITY_MASK:
                heights[
                    ~skp_quality_mask[start:stop]
                ] = np.nan

            maps[drop_db][start:stop] = heights.astype(
                np.float32
            )

    return maps


# =========================================================
# 4. COMPUTE CANOPY HEIGHT MAPS
# =========================================================
fourier_maps = canopy_height_maps_db(
    cubeF, "Fourier",
    [DB_THRESHOLDS["Fourier"]],
    CHUNK_SIZE
)

capon_maps = canopy_height_maps_db(
    cubeC, "Capon",
    [DB_THRESHOLDS["Capon"]],
    CHUNK_SIZE
)

music_drops = [DB_THRESHOLDS["MUSIC"]]

if PLOT_MUSIC_NO_THRESHOLD:
    music_drops.append(None)

music_maps = canopy_height_maps_db(
    cubeM, "MUSIC", music_drops, CHUNK_SIZE
)

fourier_canopy_map = fourier_maps[DB_THRESHOLDS["Fourier"]]
capon_canopy_map = capon_maps[DB_THRESHOLDS["Capon"]]
music_canopy_map = music_maps[DB_THRESHOLDS["MUSIC"]]

if PLOT_MUSIC_NO_THRESHOLD:
    music_no_threshold_map = music_maps[None]

# =========================================================
# 5. INTERPOLATE GEDI ONTO TOMOSAR GRID
# =========================================================
# Same spatial coordinates for both sides of each comparison.

gedi_grid = np.ma.filled(
    np.ma.asarray(gedi_height, dtype=np.float64),
    np.nan
)

gedi_grid[
    ~np.isfinite(gedi_grid) | (gedi_grid < 0)
] = np.nan

gedi_az_interp = np.asarray(gedi_az, dtype=float)
gedi_rg_interp = np.asarray(gedi_rg, dtype=float)

if gedi_grid.shape != (
    len(gedi_az_interp), len(gedi_rg_interp)
):
    raise ValueError("GEDI grid dimensions do not match its axes.")

# RegularGridInterpolator needs increasing coordinates.
if gedi_az_interp[0] > gedi_az_interp[-1]:
    gedi_az_interp = gedi_az_interp[::-1]
    gedi_grid = gedi_grid[::-1, :]

if gedi_rg_interp[0] > gedi_rg_interp[-1]:
    gedi_rg_interp = gedi_rg_interp[::-1]
    gedi_grid = gedi_grid[:, ::-1]

gedi_interpolator = RegularGridInterpolator(
    (gedi_az_interp, gedi_rg_interp),
    gedi_grid,
    method="linear",
    bounds_error=False,
    fill_value=np.nan
)

az_mesh, rg_mesh = np.meshgrid(
    tomo_az, tomo_rg, indexing="ij"
)

gedi_on_tomo = gedi_interpolator(
    np.column_stack([
        az_mesh.ravel(),
        rg_mesh.ravel()
    ])
).reshape(cubeF.shape[:2])

# =========================================================
# 6. ORGANIZE COMPARISONS
# =========================================================
comparisons = {
    "Fourier": [
        (
            fourier_canopy_map,
            f"Fourier — {DB_THRESHOLDS['Fourier']:.2f} dB",
        )
    ],
    "Capon": [
        (
            capon_canopy_map,
            f"Capon — {DB_THRESHOLDS['Capon']:.2f} dB",
        )
    ],
    "MUSIC": [
        (
            music_canopy_map,
            f"MUSIC — {DB_THRESHOLDS['MUSIC']:.2f} dB",
        )
    ],
}

if PLOT_MUSIC_NO_THRESHOLD:
    comparisons["MUSIC"].append(
        (
            music_no_threshold_map,
            "MUSIC — No dB threshold"
        )
    )

# =========================================================
# 7. CREATE SIX FIGURES
# =========================================================
figures = {}

for algorithm, variants in comparisons.items():

    nrows = len(variants)

    # -----------------------------------------------------
    # 7.1 2D MAPS — ALGORITHM VS GEDI
    # -----------------------------------------------------
    fig2d, axes2d = plt.subplots(
        nrows,
        2,
        figsize=(14, 5.5 * nrows),
        constrained_layout=True
    )

    axes2d = np.atleast_2d(axes2d)

    for row, (height_map, label) in enumerate(variants):

        for col, (data, title) in enumerate([
            (height_map, label),
            (gedi_on_tomo, "GEDI LiDAR")
        ]):

            ax = axes2d[row, col]

            im = ax.pcolormesh(
                tomo_rg / 1000,
                tomo_az / 1000,
                data,
                shading="nearest",
                cmap=HEIGHT_CMAP,
                vmin=HEIGHT_MIN,
                vmax=HEIGHT_MAX,
                rasterized=True
            )

            ax.set_title(title)
            ax.set_xlabel("Slant range [km]")
            ax.set_ylabel("Azimuth [km]")

            ax.set_xlim(
                tomo_rg[0] / 1000,
                tomo_rg[-1] / 1000
            )
            ax.set_ylim(
                tomo_az[-1] / 1000,
                tomo_az[0] / 1000
            )

    fig2d.colorbar(
        im,
        ax=axes2d.ravel().tolist(),
        label="Canopy height [m]",
        shrink=0.85
    )

    figures[f"{algorithm.lower()}_2d_vs_gedi"] = fig2d

    # -----------------------------------------------------
    # 7.2 3D SURFACES — ALGORITHM VS GEDI
    # -----------------------------------------------------
    fig3d = plt.figure(
        figsize=(16, 7 * nrows),
        constrained_layout=True
    )

    axes3d = []

    for row, (height_map, label) in enumerate(variants):

        for col, (data, title) in enumerate([
            (height_map, label),
            (gedi_on_tomo, "GEDI LiDAR")
        ]):

            ax = fig3d.add_subplot(
                nrows,
                2,
                row * 2 + col + 1,
                projection="3d"
            )

            # Existing plotting helper from setup
            plot_height_surface(
                ax,
                data,
                tomo_az,
                tomo_rg,
                title
            )

            axes3d.append(ax)

    # Existing shared colorbar helper
    add_height_colorbar(
        fig3d,
        axes3d
    )

    figures[f"{algorithm.lower()}_3d_vs_gedi"] = fig3d

# =========================================================
# 8. SUMMARY AND METADATA
# =========================================================
print("Canopy height comparisons")
print("-------------------------")

for algorithm, variants in comparisons.items():

    for height_map, label in variants:

        valid = np.isfinite(height_map)

        print(
            f"{label:30s}: "
            f"{valid.sum():,} / {height_map.size:,} "
            f"pixels retained"
        )

print(
    f"GEDI LiDAR: "
    f"{np.isfinite(gedi_on_tomo).sum():,} / "
    f"{gedi_on_tomo.size:,} valid pixels"
)

selection = {
    "algorithms": ["Fourier", "Capon", "MUSIC"],

    "db_thresholds": DB_THRESHOLDS,

    "threshold_definition": (
        "First downward dB crossing toward higher elevation "
        "from the selected spectral peak"
    ),

    "peak_selection": {
        "Fourier": "Global maximum",
        "Capon": "Global maximum",
        "MUSIC": (
            "Highest-elevation local peak detected using "
            "scipy.signal.find_peaks, subject to elevation limit"
        ),
    },

    "music_max_peak_height_m": MUSIC_MAX_PEAK_HEIGHT_M,
    "music_no_threshold_comparison": PLOT_MUSIC_NO_THRESHOLD,

    "gedi_height_filter_applied": bool(
        APPLY_GEDI_HEIGHT_FILTER
    ),
    "gedi_global_height_threshold_m": float(
        HEIGHT_THRESHOLD
    ),

    "skp_quality_mask_applied": bool(
        APPLY_SKP_QUALITY_MASK
    ),

    "gedi_resampling": (
        "Linear interpolation onto TomoSAR spatial grid"
    ),

    "display_height_limits_m": [
        float(HEIGHT_MIN),
        float(HEIGHT_MAX)
    ],

    "surface_max_points_per_axis": int(
        SURFACE_MAX_POINTS
    ),
}

# =========================================================
# 9. SAVE FIGURES
# =========================================================
# Existing helper uses outputs_dir() from config/local.toml.

run_dir = save_plot_set(
    "canopy_threshold_vs_gedi",
    figures,
    selection=selection
)

print(f"\nCreated {len(figures)} comparison figures.")

# =========================================================
# 10. DISPLAY AND CLOSE
# =========================================================
plt.show()

for fig in figures.values():
    plt.close(fig)